# example of read midas FLASH file
```
                +-------------------+
                |     Web (mhttpd)  |
                +---------+---------+
                          |
                          v
+---------+      +------------------+      +-----------+
|Frontend | ---> |   Event Buffer   | ---> |  Logger   |
+---------+      +------------------+      +-----------+
      |                    |
      |                    v
      |              +-----------+
      |              | Analyzer  |
      |              +-----------+
      |
      v
+-------------------+
|        ODB        |
+-------------------+
```
-  Frontend: Reads hardware → builds events (banks ).
-  Event Buffer (shared memory): Frontend sends events to buffer.
-  Consumers read buffer: Logger → writes data to disk, Analyzer → processes data online
-  ODB (Online Database): Stores configuration, run state, parameters (used by all components).
-  → Hardware → Frontend → Event Buffer → Logger / Analyzer → Disk
```
Event
 ├── Bank "SPEC" → SPECTUM Digitalizer values
 ├── Bank "MERC" → Temperature values
 ├── Bank "TEFP" → Temperature values
 ├── Bank "TEFP" → Temperature values
 └── Bank "... " → ...
```
- Setup storage flash-data and varibile excuting:<br>
  /cvmfs/mazzitel-personalrepo.infn.it/FLASH/config/package_setup.sh 

In [ ]:
import midas.file_reader
import os
run     = int(input("run number [int] "))


home = os.path.expanduser("~")
path    = home+'/flash-data/QUAX/TEST'
fname = ('/run%05d.mid.gz' % run)
mf = midas.file_reader.MidasFile(path+fname)

# returns an object containing the Begin-of-Run ODB snapshot, with the full ODB tree available as a nested dictionary in its .data attribute.  #
odb = mf.get_bor_odb_dump().data

try:
    Run_description   = odb['Experiment']['Run Parameters']['Run description']
    Run_number = odb["Runinfo"]["Run number"]
    print('Run_description: ', Run_description)
    print('run_number: ', Run_number)
except:
    print('WARNING: no run description')


In [ ]:
# fllib (FLASH library) are a set of useful tool under development for the FALSH eexperiment
import fllib as fl
# lettura logbook
# https://docs.google.com/spreadsheets/d/1dBHc4fwQgmx092ohra6Y-ueF_BpOPVk26BNhi5dCRnM/
logbook =  fl.panda_from_gspreadsheet(key='1dBHc4fwQgmx092ohra6Y-ueF_BpOPVk26BNhi5dCRnM', sheet_name='log')
print()
print(">> description:", logbook[(logbook.run==150)].description.values[0])
print(">> table and value:")
logbook.tail(1)

In [ ]:
%matplotlib inline
import numpy as np
import midas.file_reader
from datetime import datetime
import os

# load file
run     = int(input("run number [int] "))
home = os.path.expanduser("~")
path    = home+'/flash-data/QUAX/TEST'   
fname = ('/run%05d.mid.gz' % run)
mf = midas.file_reader.MidasFile(path+fname)

# get odb dump and retrive info  #######
odb = mf.get_bor_odb_dump().data

try:
    Run_description   = odb['Experiment']['Run Parameters']['Run description']    
    print('Run_description: ', Run_description)

except:
    print('WARNING: no run description')


for event in mf:
    if event.header.is_midas_internal_event():
        print("Saw a special event")
        continue

    bank_names = ", ".join(b.name for b in event.banks.values())
    event_number = event.header.serial_number
    event_time = datetime.fromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S')
    if event_number % 1000==0:
        print("Event # %s of type ID %s contains banks %s" % (event_number, event.header.event_id, bank_names))
        print("Received event with timestamp %s containing banks %s" % (event.header.timestamp, bank_names))
        print("Event # %s at %s, banks %s" % (event_number, datetime.utcfromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S'), bank_names))

    for bank_name, bank in event.banks.items():
        if ('SPEC' in bank_name): 
            ########################### put your code ##################################
            # example decode Spectrum dicgitalizer
            # ------------
            inputRange = 5
            Nch = 8
            fs = 5e6
            # ------------
            u = np.asarray(event.banks['SPEC'].data, dtype=np.uint16)  # 0..65535
            s = u.view(np.int16)                                       # -32768..32767
            Nsamp = s.size // Nch
            frames = s[:Nsamp * Nch].reshape(Nsamp, Nch)  # [time, ch]
            volt = frames.astype(np.float64) * (2.0 * inputRange / 65536.0)
            t = np.arange(Nsamp) / fs  # [s]
            if event_number % 100==0: 
                # print first samples  of the buffer every 100 events just for debug
                print("-----------------------")
                print(event.header.timestamp, event_number,"data:", event.banks['SPEC'].data[:10])
                print("> ch0, first 5 sample >",volt[:5,0]) # print first 5 sample of channel 0


            #############################################################################

#
#        if ('MERC' in bank_name): 
#            ########################### put your code ##################################
#            
#        .....
#
#        if ('...' in bank_name): 
#            ########################### put your code ##################################
#            


print('DONE')
